# 画像から動画を作る(Google Colab・無料GPU用)
キャラクターの画像1枚から、動く動画(数秒)を作ります。**未検証**: このリポジトリの環境にはGPUが無いので、動作確認はまだできていません。最初に1回だけ試して、うまく動かなければ教えてください。

## 使い方(初心者向け)
1. 上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存。
2. 下のセルを、上から順に ▶ ボタンで実行する(各セルは数分かかります)。
3. 画像のアップロードの所で、キャラクターの画像(例: `anime_07_B.png`)を選ぶ。
4. 最後に、動画(mp4)がダウンロードできます。

## 使うAI
- Stable Video Diffusion(SVD-XT): 1枚の絵から約3〜4秒の動画。ライセンスの条件があるので、商用利用は必ず公式ページで確認してください。
- Hugging Face のアカウントとトークン(読み取り用・無料)が必要です。SVDのページで「利用条件に同意」を押してください。

In [ ]:
!pip -q install diffusers transformers accelerate imageio imageio-ffmpeg

In [ ]:
from huggingface_hub import login
login()   # 表示される入力欄に、Hugging Face の「読み取り用トークン」を貼り付ける

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt", torch_dtype=torch.float16, variant="fp16")
pipe.enable_model_cpu_offload()   # メモリが足りない時の工夫(少し遅くなる)

In [ ]:
from google.colab import files
from PIL import Image
up = files.upload()
name = list(up.keys())[0]
img = Image.open(name).convert("RGB")
# SVD は 1024x576 が基本。縦動画にしたいので 576x1024 にそろえる(中央を切り出して拡大)
W, H = 576, 1024
scale = max(W / img.width, H / img.height)
img = img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS)
l, t = (img.width - W) // 2, (img.height - H) // 2
img = img.crop((l, t, l + W, t + H))
img

In [ ]:
import torch
from diffusers.utils import export_to_video
gen = torch.manual_seed(1)
# motion_bucket_id: 大きいほど大きく動く(127が標準、動きが足りなければ160〜200)。noise_aug_strength: 小さいほど元の絵に忠実
frames = pipe(img, decode_chunk_size=2, generator=gen, motion_bucket_id=127, noise_aug_strength=0.02,
              num_frames=25, height=H, width=W).frames[0]
export_to_video(frames, "clip.mp4", fps=7)

In [ ]:
# なめらかにする(7fps → 28fps)。ffmpeg の補間を使う
!ffmpeg -y -loglevel error -i clip.mp4 -vf "minterpolate=fps=28:mi_mode=mci,scale=1080:1920:flags=lanczos" -pix_fmt yuv420p clip_smooth.mp4
from google.colab import files
files.download("clip_smooth.mp4")

## 次のステップ(このあと、Claude に渡す)
`clip_smooth.mp4` を、チャットにアップロードしてください。Claude が、声・字幕・BGM・つなぎを重ねて、完成した動画にします。
動きが足りない/大きすぎるときは、`motion_bucket_id` を変えて、もう一度実行してください。